# 🔬 Análise Completa de ML — Clima × Dengue (Região Sul)

Este notebook realiza:
1. **EDA** — Análise exploratória com distribuições, correlações e sazonalidade
2. **Seleção** — 3 municípios epidêmicos representativos
3. **Baseline** — Modelo ingenuamente sazonal (média histórica do mês)
4. **Prophet** — Séries temporais com regressores climáticos
5. **Comparação** — Tabela de métricas (MAE, RMSE, R²) lado a lado

## 1. Setup e Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
import sys
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
plt.rcParams['figure.dpi'] = 120

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

print(f'Raiz do projeto: {BASE_DIR}')

Raiz do projeto: C:\Users\ruiga\OneDrive\Documents\Projeto_Clima_X_Dengue


## 2. Carregamento do Painel Completo (1.190 municípios × 10 anos)

In [2]:
from src.processamento.motor_lote_modelagem_v2_por_latitude import build_base_panel

painel, lookup = build_base_panel(BASE_DIR)
print(f'Registros: {len(painel):,}')
print(f'Municipios: {painel["code"].nunique()}')
print(f'Colunas: {painel.columns.tolist()}')
painel.head()

16:51:54 [INFO] build_base_panel → fonte canônica: output/csvs_latitude/ (1191 municípios esperados)


Registros: 142,800
Municipios: 1190
Colunas: ['code', 'year', 'month', 'dengue_cases', 'latitude', 'city_name', 'lat_faixa_id', 'temp_media', 'precipitacao', 'enso']


,code,year,month,dengue_cases,latitude,city_name,lat_faixa_id,temp_media,precipitacao,enso
0,410010,2015,1,0.0,-23.307179,Abatiá,2,25.2,164.2,0.5
1,410010,2015,2,1.0,-23.307179,Abatiá,2,23.2,112.3,0.5
2,410010,2015,3,8.0,-23.307179,Abatiá,2,22.2,128.2,0.5
3,410010,2015,4,5.0,-23.307179,Abatiá,2,19.6,85.2,0.7
4,410010,2015,5,7.0,-23.307179,Abatiá,2,18.8,126.4,0.9


---
## 3. EDA — Análise Exploratória de Dados
### 3.1 Distribuição dos Casos de Dengue

In [3]:
# Agregar total de casos por municipio (2015-2024)
casos_por_cidade = painel.groupby(['code', 'city_name'])['dengue_cases'].sum().reset_index()
casos_por_cidade = casos_por_cidade.sort_values('dengue_cases', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma
axes[0].hist(casos_por_cidade['dengue_cases'], bins=50, color='coral', edgecolor='black', alpha=0.8)
axes[0].set_title('Distribui\u00e7\u00e3o: Total de Casos por Munic\u00edpio (2015-2024)', fontsize=13)
axes[0].set_xlabel('Total de Casos')
axes[0].set_ylabel('N\u00ba de Munic\u00edpios')
axes[0].axvline(casos_por_cidade['dengue_cases'].median(), color='blue', linestyle='--', label=f'Mediana: {casos_por_cidade["dengue_cases"].median():.0f}')
axes[0].legend()

# Top 20
top20 = casos_por_cidade.head(20)
axes[1].barh(top20['city_name'], top20['dengue_cases'], color='crimson', edgecolor='black')
axes[1].set_title('Top 20 Munic\u00edpios com Mais Casos (2015-2024)', fontsize=13)
axes[1].set_xlabel('Total de Casos')
axes[1].invert_yaxis()
for i, v in enumerate(top20['dengue_cases']):
    axes[1].text(v + 500, i, f'{v:,.0f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(BASE_DIR / 'output' / 'eda_distribuicao_casos.png', bbox_inches='tight')
plt.show()

print(f'\nResumo estat\u00edstico de casos por munic\u00edpio:')
print(casos_por_cidade['dengue_cases'].describe())


Resumo estatístico de casos por município:
count      1190.000000
mean       2036.389076
std        9307.007542
min           0.000000
25%          16.000000
50%         224.500000
75%        1242.500000
max      176122.000000
Name: dengue_cases, dtype: float64


### 3.2 Sazonalidade Mensal (Padrão Epidemiológico)

In [4]:
# Media mensal de casos por municipio (sazonalidade)
sazonal = painel.groupby('month')['dengue_cases'].agg(['mean', 'median', 'std']).reset_index()
meses_nomes = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(sazonal['month'], sazonal['mean'], color='orangered', edgecolor='black', alpha=0.85, label='M\u00e9dia')
ax.plot(sazonal['month'], sazonal['median'], 'ko-', label='Mediana', linewidth=2)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_nomes)
ax.set_title('Sazonalidade: M\u00e9dia Mensal de Casos de Dengue por Munic\u00edpio', fontsize=14)
ax.set_ylabel('Casos / Munic\u00edpio / M\u00eas')
ax.legend()
plt.tight_layout()
plt.savefig(BASE_DIR / 'output' / 'eda_sazonalidade_mensal.png', bbox_inches='tight')
plt.show()

print('\nM\u00eas de pico:', meses_nomes[sazonal['mean'].idxmax() - 1])


Mês de pico: Fev


### 3.3 Correlações Clima × Dengue

In [5]:
from scipy.stats import spearmanr

# Agrupar por mes (serie temporal regional)
mensal = painel.groupby(['year', 'month']).agg({
    'dengue_cases': 'sum',
    'temp_media': 'mean',
    'precipitacao': 'mean',
    'enso': 'mean'
}).reset_index()

variaveis = ['temp_media', 'precipitacao', 'enso']
nomes = ['Temperatura M\u00e9dia', 'Precipita\u00e7\u00e3o', '\u00cdndice ONI (ENSO)']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (var, nome) in enumerate(zip(variaveis, nomes)):
    mask = mensal[var].notna() & mensal['dengue_cases'].notna()
    x = mensal.loc[mask, var]
    y = mensal.loc[mask, 'dengue_cases']
    rho, pval = spearmanr(x, y)
    
    axes[i].scatter(x, y, alpha=0.6, c='steelblue', edgecolors='black', s=50)
    z = np.polyfit(x, y, 1)
    axes[i].plot(np.sort(x), np.polyval(z, np.sort(x)), 'r--', linewidth=2)
    axes[i].set_title(f'{nome}\n\u03c1={rho:.3f}  p={pval:.4f}', fontsize=12)
    axes[i].set_xlabel(nome)
    axes[i].set_ylabel('Total Casos/M\u00eas (Regi\u00e3o Sul)')

plt.suptitle('Correla\u00e7\u00e3o de Spearman: Clima vs Dengue (Agrega\u00e7\u00e3o Mensal Regional)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'output' / 'eda_correlacoes_clima_dengue.png', bbox_inches='tight')
plt.show()

---
## 4. Seleção de 3 Municípios Epidêmicos

Escolhemos municípios com **surtos reais significativos** para testar se o modelo consegue capturar picos.

In [6]:
# Selecionar 3 cidades com mais casos
top3_codes = casos_por_cidade.head(3)['code'].tolist()
top3_names = casos_por_cidade.head(3)['city_name'].tolist()

print('Munic\u00edpios selecionados para teste:')
for i, (code, name) in enumerate(zip(top3_codes, top3_names), 1):
    total = casos_por_cidade[casos_por_cidade['code'] == code]['dengue_cases'].values[0]
    print(f'  {i}. {name.upper()} (cod: {code}) - {total:,.0f} casos totais')

Municípios selecionados para teste:
  1. JOAÇABA (cod: 420900) - 176,122 casos totais
  2. LOBATO (cod: 411360) - 165,399 casos totais
  3. FORMOSA DO OESTE (cod: 410820) - 111,011 casos totais


## 5. Funções de Modelagem

Definimos aqui as funções reutilizáveis para treinar e avaliar os modelos.

In [7]:
from prophet import Prophet

def preparar_cidade(painel, code):
    """Isola municipio, cria data e lags."""
    df = painel[painel['code'] == code].copy()
    df['data'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2))
    df = df.sort_values('data').reset_index(drop=True)
    
    # Lags de 3 meses
    df['temp_lag3'] = df['temp_media'].shift(3)
    df['prec_lag3'] = df['precipitacao'].shift(3)
    df['enso_lag3'] = df['enso'].shift(3)
    
    df = df.dropna(subset=['temp_lag3', 'prec_lag3', 'enso_lag3']).copy()
    return df

def modelo_baseline_sazonal(treino, teste):
    """Baseline: media historica do mesmo mes."""
    media_mensal = treino.groupby('month')['dengue_cases'].mean()
    predicoes = teste['month'].map(media_mensal).fillna(0).values
    return np.clip(predicoes, 0, None)

def modelo_prophet(treino, teste):
    """Prophet com regressores climaticos (lag3)."""
    df_treino = treino.rename(columns={'data': 'ds', 'dengue_cases': 'y'})
    
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                seasonality_mode='multiplicative')
    m.add_regressor('temp_lag3')
    m.add_regressor('prec_lag3')
    m.add_regressor('enso_lag3')
    m.fit(df_treino)
    
    futuro = teste[['data', 'temp_lag3', 'prec_lag3', 'enso_lag3']].copy()
    futuro = futuro.rename(columns={'data': 'ds'})
    previsoes = m.predict(futuro)
    return np.clip(previsoes['yhat'].values, 0, None)

def avaliar(y_real, y_pred):
    """Calcula metricas de avaliacao."""
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}

print('Funcoes de modelagem definidas.')

Funcoes de modelagem definidas.


---
## 6. Treinamento e Comparação (Baseline vs Prophet)

Para cada município:
- **Treino**: 2015–2022 (histórico)
- **Teste**: 2023–2024 (dados nunca vistos)

In [8]:
resultados = []

fig, axes = plt.subplots(3, 1, figsize=(16, 18))

for idx, (code, name) in enumerate(zip(top3_codes, top3_names)):
    print(f'\n{"="*60}')
    print(f'Processando: {name.upper()} (cod: {code})')
    print(f'{"="*60}')
    
    df = preparar_cidade(painel, code)
    
    # Split temporal
    treino = df[df['data'].dt.year <= 2022].copy()
    teste = df[df['data'].dt.year > 2022].copy()
    
    y_real = teste['dengue_cases'].values
    
    print(f'  Treino: {len(treino)} meses | Teste: {len(teste)} meses')
    print(f'  Casos no periodo de teste: {y_real.sum():,.0f}')
    
    # --- Baseline ---
    pred_base = modelo_baseline_sazonal(treino, teste)
    met_base = avaliar(y_real, pred_base)
    print(f'  Baseline: MAE={met_base["MAE"]:.1f}  RMSE={met_base["RMSE"]:.1f}  R2={met_base["R2"]:.3f}')
    
    # --- Prophet ---
    pred_prophet = modelo_prophet(treino, teste)
    met_prophet = avaliar(y_real, pred_prophet)
    print(f'  Prophet:  MAE={met_prophet["MAE"]:.1f}  RMSE={met_prophet["RMSE"]:.1f}  R2={met_prophet["R2"]:.3f}')
    
    # Guardar resultados
    resultados.append({'Municipio': name.upper(), 'Modelo': 'Baseline Sazonal', **met_base})
    resultados.append({'Municipio': name.upper(), 'Modelo': 'Prophet + Clima', **met_prophet})
    
    # --- Grafico ---
    ax = axes[idx]
    ax.plot(teste['data'], y_real, 'k-o', linewidth=2.5, markersize=6, label='Dengue REAL', zorder=5)
    ax.plot(teste['data'], pred_base, 's--', color='gray', linewidth=1.5, markersize=5, label=f'Baseline (R\u00b2={met_base["R2"]:.2f})', alpha=0.7)
    ax.plot(teste['data'], pred_prophet, 'x-', color='dodgerblue', linewidth=2, markersize=7, label=f'Prophet (R\u00b2={met_prophet["R2"]:.2f})')
    
    ax.set_title(f'{name.upper()} \u2014 Previs\u00e3o 2023-2024 (Dados Nunca Vistos)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Casos de Dengue / M\u00eas')
    ax.legend(fontsize=11, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Data')
plt.suptitle('Compara\u00e7\u00e3o: Baseline Sazonal vs Prophet + Clima (3 Munic\u00edpios Epid\u00eamicos)', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(BASE_DIR / 'output' / 'ml_comparacao_baseline_vs_prophet.png', bbox_inches='tight')
plt.show()


Processando: JOAÇABA (cod: 420900)
  Treino: 93 meses | Teste: 24 meses
  Casos no periodo de teste: 129,248
  Baseline: MAE=4884.2  RMSE=8706.6  R2=-0.235


16:52:01 [INFO] Chain [1] start processing


16:52:02 [INFO] Chain [1] done processing


16:52:02 [INFO] Chain [1] start processing


  Prophet:  MAE=2947.6  RMSE=5790.9  R2=0.454

Processando: LOBATO (cod: 411360)
  Treino: 93 meses | Teste: 24 meses
  Casos no periodo de teste: 97,582
  Baseline: MAE=3409.1  RMSE=5776.7  R2=-0.150


16:52:03 [INFO] Chain [1] done processing


16:52:03 [INFO] Chain [1] start processing


  Prophet:  MAE=4033.3  RMSE=6122.3  R2=-0.292

Processando: FORMOSA DO OESTE (cod: 410820)
  Treino: 93 meses | Teste: 24 meses
  Casos no periodo de teste: 59,842
  Baseline: MAE=1989.1  RMSE=4514.5  R2=-0.026


16:52:04 [INFO] Chain [1] done processing


  Prophet:  MAE=2610.3  RMSE=5092.2  R2=-0.305


---
## 7. Tabela Comparativa Final de Métricas

In [9]:
df_resultados = pd.DataFrame(resultados)
df_resultados['MAE'] = df_resultados['MAE'].round(1)
df_resultados['RMSE'] = df_resultados['RMSE'].round(1)
df_resultados['R2'] = df_resultados['R2'].round(3)

# Salvar CSV
df_resultados.to_csv(BASE_DIR / 'output' / 'ml_metricas_comparacao.csv', index=False)

print('\n' + '='*70)
print('TABELA COMPARATIVA DE METRICAS - BASELINE vs PROPHET')
print('='*70)
print(df_resultados.to_string(index=False))
print('\nArquivo salvo em: output/ml_metricas_comparacao.csv')
print('\nInterpretacao:')
print('  MAE  = Erro Medio Absoluto (menor = melhor)')
print('  RMSE = Raiz do Erro Quadratico Medio (penaliza erros grandes)')
print('  R2   = Coeficiente de Determinacao (1.0 = perfeito, <0 = pior que media)')


TABELA COMPARATIVA DE METRICAS - BASELINE vs PROPHET
       Municipio           Modelo    MAE   RMSE     R2
         JOAÇABA Baseline Sazonal 4884.2 8706.6 -0.235
         JOAÇABA  Prophet + Clima 2947.6 5790.9  0.454
          LOBATO Baseline Sazonal 3409.1 5776.7 -0.150
          LOBATO  Prophet + Clima 4033.3 6122.3 -0.292
FORMOSA DO OESTE Baseline Sazonal 1989.1 4514.5 -0.026
FORMOSA DO OESTE  Prophet + Clima 2610.3 5092.2 -0.305

Arquivo salvo em: output/ml_metricas_comparacao.csv

Interpretacao:
  MAE  = Erro Medio Absoluto (menor = melhor)
  RMSE = Raiz do Erro Quadratico Medio (penaliza erros grandes)
  R2   = Coeficiente de Determinacao (1.0 = perfeito, <0 = pior que media)


## 8. Conclusões e Próximos Passos

### Interpretação dos Resultados
- **R² > 0.5**: O modelo captura bem os picos epidêmicos
- **R² entre 0 e 0.5**: Captura tendências mas erra a magnitude dos surtos
- **R² < 0**: O modelo é pior que simplesmente usar a média

### Próximos passos sugeridos
1. Testar Prophet com `changepoint_prior_scale` ajustado (mais flexível para surtos)
2. Adicionar variáveis extras: população, altitude, urbanização
3. Treinar em lote (todos os 1.190 municípios) e gerar mapa de acurácia
4. Comparar com XGBoost e LightGBM para modelos não-temporais